# PerioBench quickstart: diagnose gum disease from a dental note

[PerioBench](https://healthtransformationlab.org) asks whether rules and AI can tell, from what a clinician wrote,
whether a patient is **healthy**, has **gingivitis**, or has **periodontitis stage I to IV** (2017 AAP/EFP
classification). Finding it accurately in the record is how gum disease gets counted, followed and treated.

In about a minute on a CPU, this notebook:
1. loads the notes (3,484 to train on, 240 to test on);
2. trains a simple baseline: word counts and logistic regression;
3. scores it on the test notes, with a 95% interval;
4. puts it beside every method on the published leaderboard.

**Everything here is synthetic**: no real patients, scored against the generator's truth, not clinical
performance. A personal learning project, not affiliated with any organization.
Data: [Hugging Face](https://huggingface.co/datasets/healthtransformationlab/perio-bench-notes) ·
[Kaggle](https://www.kaggle.com/datasets/jcblearning/perio-bench-notes) ·
Code: [GitHub](https://github.com/healthtransformationlab/health-transformation-lab-nlp)

## 1. Load the notes
On Kaggle the attached datasets are read from disk; anywhere else (Colab, your laptop) they come from Hugging Face. Set `PERIO_BENCH_DIR` to read a local copy instead.

In [ ]:
import glob
import math
import os

import pandas as pd

HF = "hf://datasets/healthtransformationlab"


def find(*names, hf_path):
    """The first file found: an attached Kaggle dataset, a local copy, else Hugging Face."""
    for name in names:
        for root in (os.environ.get("PERIO_BENCH_DIR"), "/kaggle/input"):
            if root:
                hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
                if hits:
                    return hits[0]
    return hf_path


def load_test():
    path = find("test.csv", "review_key.csv", hf_path=f"{HF}/perio-bench-notes/data/test.csv")
    test = pd.read_csv(path)   # Kaggle's review_key.csv names the labels truth_status, truth_grade, truth_extent
    return test.rename(columns={"truth_status": "label", "truth_grade": "grade", "truth_extent": "extent"})


train = pd.read_csv(find("train.csv", hf_path=f"{HF}/perio-bench-notes/data/train.csv"))
test = load_test()
print(f"{len(train):,} training notes, {len(test):,} test notes (synthetic)")

## 2. Look at the data

In [ ]:
print(train.label.value_counts().to_string())
print()
print(train.note_text.iloc[0][:600])

## 3. Train a baseline
Word and word-pair counts (TF-IDF) and logistic regression: the leaderboard's `tfidf_logreg`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), LogisticRegression(max_iter=2000))
model.fit(train.note_text, train.label)
test["pred"] = model.predict(test.note_text)

## 4. Score it on the 240 test notes

In [ ]:
def wilson(k, n, z=1.96):
    """95% interval for a share (Wilson)."""
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


correct = int((test.pred == test.label).sum())
accuracy = correct / len(test)
lo, hi = wilson(correct, len(test))
print(f"Your baseline: {correct} of {len(test)} correct, {100 * accuracy:.1f}% "
      f"(95% CI {100 * lo:.1f}-{100 * hi:.1f}) on synthetic notes")
print()
print(pd.crosstab(test.label, test.pred, rownames=["truth"], colnames=["predicted"]))

## 5. Beside the leaderboard
Every method published in [perio-bench-results](https://huggingface.co/datasets/healthtransformationlab/perio-bench-results), on the same 240 notes.

In [ ]:
try:
    board = pd.read_csv(find("ladder_vs_truth.csv", hf_path=f"{HF}/perio-bench-results/raw/ladder/ladder_vs_truth.csv"))
except Exception as e:   # no internet and no results dataset attached
    board = None
    print(f"Leaderboard not available here ({type(e).__name__}); attach perio-bench-results to compare.")
if board is not None:
    mine = pd.DataFrame([{"method": "your baseline", "accuracy": accuracy, "acc_lo": lo, "acc_hi": hi}])
    table = pd.concat([board[["method", "accuracy", "acc_lo", "acc_hi"]], mine], ignore_index=True)
    table = table.sort_values("accuracy", ascending=False).reset_index(drop=True)
    for c in ("accuracy", "acc_lo", "acc_hi"):
        table[c] = (100 * table[c]).round(1)
    print("Correct, % of 240 synthetic review notes (95% CI):")
    print(table.to_string())

## Next steps

- Swap in your own model in step 3, then rerun steps 4 and 5.
- The hardest notes leave the diagnosis to be inferred from the findings: `review_key` (on
  [Hugging Face](https://huggingface.co/datasets/healthtransformationlab/perio-bench-notes)) gives each test note's
  difficulty (L1 to L3) and what every other part of the record says.
- Where each method on the leaderboard comes from, with citations:
  [docs/algorithms.md](https://github.com/healthtransformationlab/health-transformation-lab-nlp/blob/main/docs/algorithms.md).